# TP 2 - Assistant RAG simple


---
## 0. Configuration partagée


Version RAG simple : transformer la question en vecteur, récupérer les passages les plus proches, puis répondre avec ces sources. Cette version sert de référence avant les méthodes avancées de `2_4_rag_assistant_improved.ipynb`.

**TODO — `RAGAssistant`**

Fichier à modifier : `shared/rag_utils.py`

Le notebook appelle plusieurs fois la même logique de recherche. Cette classe sert à centraliser le chargement de la base Chroma et la récupération des chunks

`RAGAssistant` : classe qui charge la base Chroma et retourne des paires `(RAGChunk, score)` via la méthode `search`

Les détails des paramètres d'entrée et de sortie sont décrits dans les docstrings de `shared/rag_utils.py`

In [1]:
from shared.config import ROOT_DIR
from shared.llm_utils import LLMRequest, run_llm
from shared.rag_utils import RAGAssistant

DATA_DIR = ROOT_DIR / "TP2_travel_planner_RAG" / "data"
VECTOR_DB_DIR = DATA_DIR / "chroma_db_rag_v1"  # choisir entre chroma_db_rag_v1 et chroma_db_rag_v2 selon la base à tester

# TODO : initialiser la classe RAGAssistant avec la bonne base vectorielle
rag_assistant = RAGAssistant(persist_dir=VECTOR_DB_DIR, top_k=10)

---
## 1. Entrée


La requête utilisateur et le prompt système sont déjà fournis.

Concrètement, il faut vérifier avant exécution :
- que le prompt interdit toute invention,
- que chaque affirmation factuelle doit être sourcée,
- que les informations manquantes sont explicitement signalées.


In [2]:
user_query = """
Je veux partir 4 jours à Rome en avril, je n'ai pas encore les dates exactes.
Propose-moi un itinéraire. Mon budget est de 200 euros pour les sorties et les restaurants.
Je veux éviter les zones trop touristiques et découvrir des lieux plus confidentiels.
"""

system_prompt = """
Tu es un assistant de planification de voyage basé sur la méthode RAG.

### Règles:
- Utiliser uniquement les faits présents dans CONTEXTE.
- Ne jamais inventer prix, dates, horaires, adresses ou transports.
- Si une information manque, écrire: "Je ne sais pas à partir du contexte fourni."
- Citer chaque affirmation factuelle au format [source - chunk id].

### Contraintes de style:
- Être concis mais précis.
- Pas de mise en forme Markdown décorative.
- Pour chaque recommandation: 1 raison courte + 1 détail pratique (horaire, lieu, budget, logistique).
- Préférer des suggestions concrètes aux phrases vagues.

### Format de réponse:
1) Résumé: réponse courte (2 à 4 phrases)
2) Plan: séquence pratique adaptée à la demande
3) Informations manquantes: liste concise des faits absents
"""

---
## 2. Récupérer le contexte


Inspecter les chunks récupérés avant génération.

Contrôles concrets:
- Pertinence: les chunks répondent-ils réellement à la requête ?
- Couverture: proviennent-ils de plusieurs sources utiles ?
- Si c'est mauvais ici, corriger le chunking/indexation, pas le prompt de génération.


In [3]:
top_chunks = rag_assistant.search(query=user_query, top_k=10)

In [4]:
print(f"Chunks récupérés : {len(top_chunks)}")
print(f"Fichiers et volume :")
source_counts = {}
for chunk, _score in top_chunks:
    source = chunk.source
    if source in source_counts:
        source_counts[source] += 1
    else:
        source_counts[source] = 1
for source, count in source_counts.items():
    print(f"  {source}: {count} chunks")
print("\n"*5)

for rank, (chunk, score) in enumerate(top_chunks, start=1):
    preview = chunk.text.replace("\n", " ")
    print(f"Chunk #{rank} | score_distance_vecteur={score:.4f} | source={chunk.source} | chunk_id={chunk.chunk_id}")
    print(preview)
    print("\n\n")

Chunks récupérés : 10
Fichiers et volume :
  rome_5_days_guide.md: 7 chunks
  rome_guide_lieux.md: 2 chunks
  italie_nord_petit_fute.md: 1 chunks






Chunk #1 | score_distance_vecteur=0.6544 | source=rome_5_days_guide.md | chunk_id=7
of Neptune (northern end) The beautiful building towering above the central fountain is the church of Sant'Agnese in Agone by Francesco Borromini and Girolamo Rainaldi  - Sit down and relax while doing some people watching  #### Conseils pratiques  Beautiful day or night  - In the 15th century, the city market was held in the square that still hosts Rome's Christmas market  **16:45–17:15**  ### Campo dei Fiori  (Piazza Campo de' Fiori, 00182 Rome)  In the evening it becomes a busy outdoor salon of carefree atmosphere  #### À savoir  Campo dei Fiori is a lively square in Rome In ancient times, the square was the open space in front of Theatre of Pompey, Rome's first stone theater and place of Julius Caesar's murder  #### À faire  Take a break here Have a 

---
## 3. Générer la réponse


Les chunks sont concaténés dans un bloc CONTEXTE, injectés dans le prompt système, puis envoyés au modèle.

Validation concrète de la sortie:
- chaque fait important doit être sourcé,
- aucune invention,
- les zones d'incertitude doivent être annoncées explicitement.


In [5]:
top_chunks = rag_assistant.search(user_query)
context = "\n\n".join(
    [f"[{chunk.source} - chunk {chunk.chunk_id}]\n{chunk.text}" for chunk, _score in top_chunks]
)
grounded_system_prompt = f"{system_prompt}\n\nCONTEXTE:\n{context}"
run_result = await run_llm(LLMRequest(system_prompt=grounded_system_prompt, user_prompt=user_query))

token_usage = {
    "input_tokens": run_result.input_tokens,
    "output_tokens": run_result.output_tokens,
    "total_tokens": run_result.total_tokens,
}

print(run_result.output)
print()
print(
    f"tokens : entrée={token_usage['input_tokens']} | "
    f"sortie={token_usage['output_tokens']} | "
    f"total={token_usage['total_tokens']}"
)


Voici une proposition d'itinéraire pour découvrir Rome en 4 jours en avril, en privilégiant des lieux moins fréquentés et en gardant à l'esprit votre budget.

1) Résumé:
Ce programme de 4 jours à Rome vous invite à explorer des quartiers pleins de charme comme le Ghetto Juif et Trastevere, tout en découvrant des sites historiques majeurs sous un angle différent. L'itinéraire équilibre la découverte culturelle avec des moments de détente dans des places animées, en tenant compte de votre budget pour les visites et repas.

2) Plan:

Jour 1: Immersion historique et artistique
- Matin : Visite du Panthéon.
    - Raison : Admirez un édifice antique exceptionnellement préservé, passé du culte païen au culte chrétien. [rome_guide_lieux.md - chunk 1]
    - Détail pratique : L'entrée est gratuite. [rome_guide_lieux.md - chunk 1]
- Après-midi : Exploration du Forum Romain et du Palatin.
    - Raison : Plongez au cœur de la vie publique de la Rome antique. [rome_guide_lieux.md - chunk 1]
    - Dé